# Power BI for Data Analysis Bootcamp

A practical, beginner-friendly course covering **26 Power BI concepts** for building data models and interactive reports. Each topic has two cells: one Markdown explanation and one Python code cell. DAX and Power Query examples are provided as `#`-commented reference snippets, with executable Python demonstrations to illustrate the underlying logic.

**Companion data:** Import `power_bi_practice_data.xlsx` into Power BI Desktop. It contains a fact table, dimension tables, a DAX reference, and a data dictionary.

**Important:** DAX and Power Query M are not Python. Their examples in this notebook are reference text, not code you run in Jupyter. Create DAX measures in Power BI Desktop and use Power Query's Advanced Editor for M code.

## Learning path
1. Data model and star schema
2. Power Query and data quality
3. DAX measures and filter context
4. Time intelligence and targets
5. Report visuals, interactions, security, and publishing
6. Final sales performance report

## 1. What Power BI Does

Power BI is a business-intelligence platform for connecting to data, transforming it, modeling relationships, defining measures, and building interactive reports.

The typical workflow is: **Get Data → Power Query → Data Model → DAX → Visuals → Publish/Refresh**. Power BI is not simply a charting tool; good models and metric definitions matter as much as the visuals.

In [ ]:
# In Power BI Desktop: Home → Get Data → Excel workbook.
# Select power_bi_practice_data.xlsx and choose FactSales, DimProduct,
# DimCustomer, DimRegion, and DimDate (also import other tables if needed).
# Use Transform Data to inspect and clean the data before loading.
data_steps = ["Connect", "Transform", "Model", "Measure", "Visualize", "Publish"]  # Represent the main workflow steps.
for step_number, step in enumerate(data_steps, start=1):  # Walk through the workflow in order.
    print(step_number, step)  # Display each stage in the process.

## 2. Understand Fact Tables and Grain

A fact table stores events or transactions. **Grain** means exactly what one row represents. In this practice dataset, one FactSales row is one sales line item, so OrderID repeats when an order contains multiple products.

This matters for metrics: `COUNTROWS(FactSales)` counts lines, not orders. Use `DISTINCTCOUNT(FactSales[OrderID])` when you need order count.

In [ ]:
# DAX measure to count sales lines:
# Sales Lines = COUNTROWS(FactSales)
# DAX measure to count unique orders:
# Order Count = DISTINCTCOUNT(FactSales[OrderID])
sample = [{"OrderID": 10, "Product": "Laptop"}, {"OrderID": 10, "Product": "Mouse"}, {"OrderID": 11, "Product": "Phone"}]  # Simulate order lines.
line_count = len(sample)  # Count fact rows.
order_count = len({row["OrderID"] for row in sample})  # Count distinct order IDs.
print("Sales lines:", line_count)  # Show the count of product lines.
print("Distinct orders:", order_count)  # Show the correct order count.

## 3. Star Schema and Relationships

A star schema has a central fact table connected to dimension tables. Dimensions describe the facts—product, customer, region, date—while the fact table stores quantities and amounts.

Create one-to-many relationships from each dimension's unique key to the matching fact column. Prefer single-direction filtering from dimensions toward the fact table unless the model requires otherwise.

In [ ]:
# Recommended relationships for the practice workbook:
# DimProduct[ProductID]  (1) -> (*) FactSales[ProductID]
# DimCustomer[CustomerID] (1) -> (*) FactSales[CustomerID]
# DimRegion[Region]      (1) -> (*) FactSales[Region]
# DimDate[Date]          (1) -> (*) FactSales[OrderDate]
# Avoid many-to-many relationships unless you understand the modeling implications.
relationships = {  # Describe the relationship keys for review.
    "Product": ("DimProduct.ProductID", "FactSales.ProductID"),  # Product dimension to fact.
    "Customer": ("DimCustomer.CustomerID", "FactSales.CustomerID"),  # Customer dimension to fact.
    "Region": ("DimRegion.Region", "FactSales.Region"),  # Region dimension to fact.
    "Date": ("DimDate.Date", "FactSales.OrderDate"),  # Date dimension to fact.
}  # Finish the relationship map.
for dimension, keys in relationships.items():  # Review each planned relationship.
    print(dimension, ":", keys[0], "→", keys[1])  # Display the one-to-many key pair.

## 4. Power Query: Clean and Transform Data

Power Query is the data-preparation layer. Use it to rename columns, set data types, remove duplicates, trim text, replace values, filter rows, and combine files before data reaches the model.

Power Query steps are repeatable transformations. Prefer doing row/column cleaning in Power Query over creating calculated columns for every cleanup operation in DAX.

In [ ]:
# Example Power Query M steps (create through the Power Query UI or Advanced Editor):
# let
#     Source = Excel.Workbook(File.Contents("power_bi_practice_data.xlsx"), null, true),
#     FactSalesSheet = Source{[Item="FactSales",Kind="Table"]}[Data],
#     ChangedTypes = Table.TransformColumnTypes(FactSalesSheet, {
#         {"OrderDate", type date}, {"Quantity", Int64.Type},
#         {"NetSales", Currency.Type}, {"DiscountPct", Percentage.Type}
#     }),
#     TrimmedRegion = Table.TransformColumns(ChangedTypes, {{"Region", Text.Trim, type text}})
# in
#     TrimmedRegion
messy_text = "  North  "  # Create a sample dirty category.
clean_text = messy_text.strip()  # Mirror a Power Query Text.Trim step.
print("Before:", repr(messy_text))  # Show extra whitespace in the original.
print("After:", clean_text)  # Show the normalized category.

## 5. Data Types and Data Quality

Assign correct data types: dates should be Date/DateTime, quantities should be whole numbers, amounts should be decimal/currency, and identifiers should usually be treated as keys rather than measures.

Check for blanks, unexpected categories, invalid quantities, duplicate keys in dimension tables, and implausible amounts. Data types also affect sorting and whether DAX time-intelligence functions work correctly.

In [ ]:
# Power BI checks to make:
# - FactSales[OrderDate] has type Date.
# - FactSales[Quantity] is a whole number.
# - FactSales[NetSales] and FactSales[Profit] are decimal/currency values.
# - DimDate[Date] is unique and contains every date in the analysis range.
orders = [{"OrderID": 1, "Quantity": 2}, {"OrderID": 2, "Quantity": 0}, {"OrderID": 3, "Quantity": 4}]  # Simulate facts to validate.
invalid_quantity = [row for row in orders if row["Quantity"] <= 0]  # Identify non-positive quantities when the business rule forbids them.
print("Invalid rows to review:", invalid_quantity)  # Display records requiring investigation.

## 6. Calculated Columns vs Measures

A **calculated column** is evaluated for each row and stored in the model. A **measure** is calculated when a visual queries it and responds to filters and slicers.

Use columns for row-level attributes or categories; use measures for totals, ratios, averages, and KPIs. For example, NetSales is already a row-level field in the practice data, while Total Sales should be a measure.

In [ ]:
# Calculated-column example, if the source does not already include NetSales:
# NetSales = FactSales[GrossSales] * (1 - FactSales[DiscountPct])
# Measure example:
# Total Sales = SUM(FactSales[NetSales])
sample_lines = [{"GrossSales": 100, "DiscountPct": 0.1}, {"GrossSales": 200, "DiscountPct": 0.05}]  # Sample source rows.
row_net_sales = [row["GrossSales"] * (1 - row["DiscountPct"]) for row in sample_lines]  # Calculate a row-level value for each row.
total_sales = sum(row_net_sales)  # Aggregate the row-level values into a total.
print("Calculated row values:", row_net_sales)  # Display each row's calculated value.
print("Measure-style total:", total_sales)  # Display the aggregation across rows.

## 7. First DAX Measures: SUM and COUNTROWS

DAX (Data Analysis Expressions) is used for measures and calculated columns in the semantic model. Measures are created with **Modeling → New measure**.

Start with simple aggregations, then add filter context and time intelligence. Name measures clearly and format amounts as currency and ratios as percentages.

In [ ]:
# Create these measures in Power BI (Modeling → New measure):
# Total Sales = SUM(FactSales[NetSales])  # Adds net sales for the current filter context.
# Total Profit = SUM(FactSales[Profit])  # Adds profit for the current filter context.
# Total Units = SUM(FactSales[Quantity])  # Adds sold quantity.
# Sales Lines = COUNTROWS(FactSales)  # Counts fact rows, not distinct orders.
sample_sales = [125.0, 250.0, 80.0, 500.0]  # Create example net sales values.
print("Total Sales:", sum(sample_sales))  # Mirror the SUM measure.
print("Sales Lines:", len(sample_sales))  # Mirror COUNTROWS on a small list.

## 8. DISTINCTCOUNT

`DISTINCTCOUNT()` counts unique values in a column under the current filter context. It is useful for distinct orders, customers, products, or sessions.

Because FactSales is at line-item grain, order IDs repeat. `DISTINCTCOUNT(FactSales[OrderID])` is typically the correct order metric; `COUNTROWS(FactSales)` measures line items.

In [ ]:
# DAX measure:
# Order Count = DISTINCTCOUNT(FactSales[OrderID])  # Count unique order headers even when they have multiple lines.
# Customer Count = DISTINCTCOUNT(FactSales[CustomerID])  # Count unique customers in current context.
order_ids = [5001, 5001, 5001, 5002, 5003, 5003]  # Simulate repeated order IDs across product lines.
print("Line count:", len(order_ids))  # Count rows in the fact table.
print("Distinct order count:", len(set(order_ids)))  # Count unique orders.

## 9. Filter Context and CALCULATE

Filter context is the set of filters affecting a visual or measure—such as selected dates, region, product, or channel. `CALCULATE()` evaluates an expression after changing the filter context.

This is one of the most important DAX concepts. A single `[Total Sales]` measure can return different values in each row of a region chart because each row applies a region filter.

In [ ]:
# Base measure:
# Total Sales = SUM(FactSales[NetSales])
# Measure that forces a North filter:
# North Sales = CALCULATE([Total Sales], DimRegion[Region] = "North")
sales_by_region = {"North": 1200, "South": 850, "East": 950, "West": 700}  # Sample grouped totals.
print("North sales:", sales_by_region["North"])  # Simulate CALCULATE with a North region filter.
print("All-region sales:", sum(sales_by_region.values()))  # Simulate the measure with no region filter.

## 10. SUMX and Row-by-Row Evaluation

`SUMX(table, expression)` iterates over a table, evaluates an expression for each row, and sums the result. It is useful when the result requires row-level arithmetic.

`SUM(FactSales[NetSales])` is simpler when NetSales already exists. If you need to calculate sales from quantity and unit price, `SUMX(FactSales, FactSales[Quantity] * FactSales[UnitPrice])` expresses the row-wise calculation.

In [ ]:
# DAX measure example:
# Gross Sales Recalculated = SUMX(FactSales, FactSales[Quantity] * FactSales[UnitPrice])
lines = [{"Quantity": 2, "UnitPrice": 100}, {"Quantity": 3, "UnitPrice": 50}]  # Create two line-item records.
line_totals = [line["Quantity"] * line["UnitPrice"] for line in lines]  # Calculate each row's extended amount.
print("Line totals:", line_totals)  # Display row-wise multiplication results.
print("SUMX-style total:", sum(line_totals))  # Sum the row-wise results.

## 11. DIVIDE and KPI Ratios

Use `DIVIDE(numerator, denominator)` for ratios. It handles a zero denominator safely and returns BLANK by default, or a specified alternate result.

It is generally safer than ordinary division for measures such as profit margin, conversion rate, average order value, and target attainment.

In [ ]:
# DAX measures:
# Profit Margin = DIVIDE([Total Profit], [Total Sales])
# Average Order Value = DIVIDE([Total Sales], [Order Count])
# Target Attainment % = DIVIDE([Total Sales], [Sales Target])
profit = 2400  # Example total profit.
sales = 12000  # Example total sales.
margin = profit / sales if sales != 0 else None  # Mirror safe division.
print("Profit margin:", f"{margin:.1%}" if margin is not None else "No sales denominator")  # Display the ratio as a percentage.

## 12. IF and SWITCH

Use `IF()` for a two-way decision and `SWITCH()` for multiple branches. `SWITCH(TRUE(), ...)` is a common pattern for ordered threshold bands.

Put the highest threshold first so a high value does not get captured by a lower rule earlier in the expression.

In [ ]:
# DAX calculated-column example:
# Revenue Band =
# SWITCH(
#     TRUE(),  # Evaluate conditions in order.
#     FactSales[NetSales] >= 1000, "High",  # Assign High to values of at least 1000.
#     FactSales[NetSales] >= 500, "Medium",  # Assign Medium to values from 500 to below 1000.
#     "Low"  # Use Low for all remaining values.
# )
def revenue_band(value):  # Define equivalent logic for a Python demonstration.
    if value >= 1000:  # Test the highest band first.
        return "High"  # Return the highest category.
    if value >= 500:  # Test the middle band.
        return "Medium"  # Return the middle category.
    return "Low"  # Return the fallback category.
print([(v, revenue_band(v)) for v in [250, 650, 1250]])  # Display sample band assignments.

## 13. RELATED and Dimension Attributes

`RELATED()` retrieves a value from a related one-side dimension table while evaluating a calculated column in the fact table. For example, it can bring product category onto each fact row.

Often you should keep attributes in dimensions and use relationships for filtering instead of copying every descriptive column into the fact table. This preserves a clean star schema.

In [ ]:
# DAX calculated column example:
# Product Category = RELATED(DimProduct[Category])
sales_lines = [{"ProductID": 101, "NetSales": 900}, {"ProductID": 102, "NetSales": 500}]  # Example fact rows.
product_category = {101: "Computers", 102: "Electronics"}  # Example dimension lookup.
enriched = [{**row, "Category": product_category[row["ProductID"]]} for row in sales_lines]  # Illustrate attaching a related dimension attribute.
print(enriched)  # Show fact rows with a category added for demonstration.

## 14. ALL and REMOVEFILTERS

`ALL()` and `REMOVEFILTERS()` can remove filters from a table or column inside `CALCULATE()`. They are frequently used for percent-of-total measures and comparisons to the unfiltered total.

Be precise about which filters you remove. Removing product filters should usually preserve date, region, and channel filters if the goal is product share within the currently selected report slice.

In [ ]:
# DAX measures:
# Sales All Products = CALCULATE([Total Sales], REMOVEFILTERS(DimProduct))
# Product Sales Share = DIVIDE([Total Sales], [Sales All Products])
product_sales = {"Laptop": 5000, "Phone": 3000, "Tablet": 2000}  # Sample product totals.
all_products_total = sum(product_sales.values())  # Simulate removing the product-level filter only.
for product, amount in product_sales.items():  # Examine each product's total.
    print(product, f"{amount / all_products_total:.1%} of total")  # Display share of all product sales.

## 15. RANKX

`RANKX()` ranks an expression evaluated over a table. It is used for top products, highest-revenue customers, and sales representatives.

When ranking products, you often remove the current product filter from the ranking table with `ALL(DimProduct[ProductName])`, while the measure still responds to filters like date and region.

In [ ]:
# DAX measure:
# Product Rank = RANKX(ALL(DimProduct[ProductName]), [Total Sales], , DESC)
product_sales = {"Laptop": 5000, "Phone": 3000, "Tablet": 3000, "Monitor": 1200}  # Sample product totals.
sorted_sales = sorted(product_sales.items(), key=lambda item: item[1], reverse=True)  # Sort products by sales descending.
for rank, (product, amount) in enumerate(sorted_sales, start=1):  # Assign a simple sequential rank for illustration.
    print(rank, product, amount)  # Display each product's rank and total; DAX RANKX supports tie options.

## 16. Date Table and Time Intelligence

Time-intelligence measures require a proper date dimension: one row per date, continuous dates across the reporting range, a relationship to the fact date, and a date table marked in the model.

Common patterns include YTD (year-to-date), MTD (month-to-date), previous-year comparisons, and year-over-year change. Use `DimDate[Date]`, not a text month label, as the date-intelligence column.

In [ ]:
# DAX time-intelligence measures:
# Sales YTD = TOTALYTD([Total Sales], DimDate[Date])
# Sales MTD = TOTALMTD([Total Sales], DimDate[Date])
# Sales Previous Year = CALCULATE([Total Sales], SAMEPERIODLASTYEAR(DimDate[Date]))
# YoY Change = [Total Sales] - [Sales Previous Year]
monthly_sales = [8000, 9200, 8700, 10500, 11200, 10900]  # Simulate six monthly totals.
running_ytd = []  # Prepare a cumulative list to illustrate YTD.
running_total = 0  # Initialize the running total.
for value in monthly_sales:  # Process months in chronological order.
    running_total += value  # Add the current month.
    running_ytd.append(running_total)  # Store the year-to-date result for the month.
print("Monthly sales:", monthly_sales)  # Display each month's revenue.
print("Running YTD totals:", running_ytd)  # Display cumulative year-to-date values.

## 17. Targets, Variance, and Attainment

Business reports often compare actual results with targets. Use a target measure at the correct grain and define whether the target is monthly, annual, regional, product-based, or otherwise.

A common set of measures is variance (`Actual − Target`) and attainment (`Actual ÷ Target`). Check that target values are not accidentally repeated and summed across fact rows.

In [ ]:
# DAX measures for a region-level target table:
# Sales Target = SUM(DimRegion[SalesTarget])
# Target Gap = [Total Sales] - [Sales Target]
# Target Attainment % = DIVIDE([Total Sales], [Sales Target])
actual_sales = 92000  # Store an example actual amount for a region.
target_sales = 90000  # Store the matching target.
gap = actual_sales - target_sales  # Calculate the absolute variance.
attainment = actual_sales / target_sales if target_sales else None  # Calculate the target attainment ratio safely.
print("Target gap:", gap)  # Display amount above or below target.
print("Target attainment:", f"{attainment:.1%}" if attainment is not None else "No target")  # Display the percentage.

## 18. Variables with VAR and RETURN

In DAX, `VAR` stores an intermediate result inside a measure and `RETURN` defines the final result. Variables improve readability and can avoid repeating the same expression.

Use meaningful names, and be aware that a variable is evaluated within the context where it is defined; it does not automatically recalculate later under a changed filter context.

In [ ]:
# DAX measure example:
# Sales with Bonus =
# VAR SalesAmount = [Total Sales]  # Store the current-context sales measure.
# VAR BonusRate = 0.05  # Define a five-percent bonus rate.
# RETURN SalesAmount * (1 + BonusRate)  # Return sales including the illustrative bonus.
sales_amount = 12000  # Provide a sample measure result.
bonus_rate = 0.05  # Store the rate in a named variable.
adjusted_amount = sales_amount * (1 + bonus_rate)  # Calculate the result using the named values.
print("Sales with illustrative bonus:", adjusted_amount)  # Display the computed result.

## 19. Build Visuals for Specific Questions

Choose visuals by the analytical question:
- Cards for headline KPIs.
- Line charts for trends over time.
- Bar/column charts for category comparisons.
- Matrix visuals for two-dimensional summaries.
- Scatter plots for relationships.
- Tables for detailed records.

Avoid clutter and avoid using pie charts for many categories. Every chart should have a clear title, meaningful labels, and understandable units.

In [ ]:
chart_guide = {  # Map business questions to appropriate Power BI visuals.
    "What are total sales and profit?": "Card visuals",  # Highlight a small number of KPIs.
    "How did sales change by month?": "Line chart",  # Show a time trend.
    "Which products sell best?": "Bar chart",  # Compare categories.
    "How do region and product interact?": "Matrix or heatmap-style conditional formatting",  # Compare two dimensions.
    "Are units and profit related?": "Scatter chart",  # Show a numeric relationship.
}  # Finish the guide.
for question, visual in chart_guide.items():  # Visit each question and recommended visual.
    print(question, "→", visual)  # Display the suggested visual type.

## 20. Slicers and Filter Context

Slicers are visible filters users can interact with—commonly Date, Region, Product Category, and Channel. Report-, page-, and visual-level filters provide other ways to restrict the data.

A well-designed report makes the active filter context clear. Use slicers sparingly and group related ones together to avoid a crowded report page.

In [ ]:
# Example report slicers:
# - DimDate[YearMonth]
# - DimRegion[Region]
# - DimProduct[Category]
# - FactSales[Channel]
filters = {"Region": "North", "Channel": "Online"}  # Simulate selected slicer values.
rows = [  # Create sample records.
    {"Region": "North", "Channel": "Online", "Sales": 1200},  # Matches both filters.
    {"Region": "North", "Channel": "Store", "Sales": 800},  # Fails the channel filter.
    {"Region": "South", "Channel": "Online", "Sales": 900},  # Fails the region filter.
]  # Finish the sample data.
visible_rows = [row for row in rows if row["Region"] == filters["Region"] and row["Channel"] == filters["Channel"]]  # Keep only rows matching both slicers.
print("Rows visible under slicers:", visible_rows)  # Display the filtered result.

## 21. Drill-Down, Drill-Through, and Tooltips

Drill-down explores a hierarchy such as Year → Quarter → Month → Day inside a visual. Drill-through navigates to a dedicated detail page filtered to a selected entity. Tooltips reveal extra context when a user hovers over a data point.

These features make reports interactive without requiring every detail on the main canvas. Keep navigation predictable and include a way to return to the overview page.

In [ ]:
# Example drill hierarchy:
# DimDate[Year] -> DimDate[Quarter] -> DimDate[MonthName] -> DimDate[Date]
# Example drill-through page: Customer Details, using DimCustomer[CustomerID] as the drill-through field.
# Example tooltip fields: Total Profit, Order Count, Profit Margin.
hierarchy = ["Year", "Quarter", "Month", "Date"]  # Define a date drill-down hierarchy for the report.
for level, field in enumerate(hierarchy, start=1):  # Enumerate the hierarchy levels.
    print(f"Level {level}: {field}")  # Display the order users can drill into.

## 22. Conditional Formatting and KPI Status

Conditional formatting draws attention to results that cross a meaningful threshold. Use data bars for magnitude, icons for status, or colors for gaps versus target.

Define the status logic explicitly and consider accessibility: pair colors with labels or icons so a red/green scheme is not the only way to distinguish outcomes.

In [ ]:
# DAX status measure example:
# Target Status =
# SWITCH(
#     TRUE(),
#     [Target Attainment %] >= 1, "On target",
#     [Target Attainment %] >= 0.9, "At risk",
#     "Below target"
# )
def target_status(attainment):  # Define sample threshold logic.
    if attainment >= 1:  # Check whether target was met.
        return "On target"  # Return the positive status.
    if attainment >= 0.9:  # Check whether attainment is within 10 percent of target.
        return "At risk"  # Return the warning status.
    return "Below target"  # Return the lower performance status.
print([(x, target_status(x)) for x in [1.05, 0.94, 0.72]])  # Display illustrative KPI statuses.

## 23. Row-Level Security (RLS)

Row-level security restricts which rows a user can see. Static roles may filter a region or business unit; dynamic RLS often uses the signed-in user's identity and a user-to-region mapping table.

Create roles in the model, test them with **View as**, and validate permissions before publishing. RLS is not a substitute for careful handling of sensitive data, and workspace roles can affect how security is applied.

In [ ]:
# Example static RLS rule for a role called North Manager:
# DimRegion[Region] = "North"
# Example dynamic rule using a user-access mapping table:
# UserAccess[Email] = USERPRINCIPALNAME()
# Relate UserAccess to the allowed region/customer key so the filter propagates correctly.
user_region_access = {"north.manager@example.com": ["North"], "south.manager@example.com": ["South"]}  # Simulate an access mapping.
active_user = "north.manager@example.com"  # Represent the signed-in user in this example.
allowed_regions = user_region_access.get(active_user, [])  # Retrieve only the regions allowed for this user.
print("Regions visible to this user:", allowed_regions)  # Display the simulated security scope.

## 24. Performance and Model Optimization

Efficient Power BI models usually have a clear star schema, appropriate data types, fewer unused columns, and simple relationships. Prefer measures for aggregations, use Power Query to remove unnecessary data early, and reduce high-cardinality columns where practical.

Avoid importing columns that are not used in analysis, creating unnecessary bidirectional relationships, or using row-by-row DAX when an existing column or simpler measure is enough. Verify performance using actual model diagnostics rather than assumptions.

In [ ]:
# Practical model review checklist:
# 1. Remove columns that are not needed for report, relationships, or measures.
# 2. Use integer keys where appropriate and correct date/numeric data types.
# 3. Keep dimension keys unique.
# 4. Prefer measures such as SUM(FactSales[NetSales]) for aggregations.
# 5. Use Performance Analyzer in Power BI Desktop to inspect slow visuals.
model_checks = {  # Summarize model hygiene checks for an example report.
    "Unused columns removed": True,  # Keep only needed fields in the model.
    "Dimension keys unique": True,  # Ensure one-side relationship keys do not repeat.
    "Date table marked": True,  # Enable dependable time-intelligence behavior.
    "Measures named and formatted": True,  # Keep fields understandable to report users.
}  # Finish the review checklist.
print(model_checks)  # Display the checklist state.

## 25. Publish, Refresh, and Governance

Publishing puts a report in the Power BI service so others can view or interact with it, depending on permissions and licensing. Refresh strategy depends on the source, connectivity, privacy, and whether the model uses Import or DirectQuery.

Before publishing, validate totals, set refresh credentials and schedules where appropriate, configure access, and document metric definitions. A report is only as trustworthy as the data refresh and model rules behind it.

In [ ]:
# Before publishing, check:
# - Does the report reconcile with known source totals?
# - Are refresh credentials configured for every source?
# - Does the refresh schedule meet the business need?
# - Are workspace, report, and dataset permissions appropriate?
# - Have RLS roles been tested using representative users?
release_checklist = ["Reconcile totals", "Configure refresh", "Review permissions", "Test RLS", "Document KPIs"]  # List release readiness steps.
for item in release_checklist:  # Walk through the pre-release checks.
    print("☐", item)  # Print an unchecked task for the analyst.

## 26. Mini Project: Sales Performance Report

Build a report that answers five questions:
1. What are total net sales, profit, units, and order count?
2. Which product categories contribute the most sales?
3. How does revenue change month to month?
4. Which regions meet their targets?
5. How do channel and customer segment affect results?

Use the supplied practice workbook. Create measures, connect the dimensions to FactSales, add cards and visuals, and finish with slicers and a clear report title. Keep the page focused on decisions, not on showing every available field.

In [ ]:
# Core DAX measures for the report:
# Total Sales = SUM(FactSales[NetSales])
# Total Profit = SUM(FactSales[Profit])
# Total Units = SUM(FactSales[Quantity])
# Order Count = DISTINCTCOUNT(FactSales[OrderID])
# Profit Margin = DIVIDE([Total Profit], [Total Sales])
# Sales YTD = TOTALYTD([Total Sales], DimDate[Date])
# Report layout suggestion:
# Top row: Total Sales, Total Profit, Profit Margin, Order Count cards.
# Left: monthly sales line chart.
# Center: sales by product/category bar chart.
# Right: region vs target chart.
# Slicers: YearMonth, Region, Category, Channel.
sample_sales = [1250, 900, 1800, 720]  # Example net sales values for a mini-summary calculation.
sample_profit = [250, 160, 360, 90]  # Example corresponding profit values.
total_sales = sum(sample_sales)  # Calculate the example sales KPI.
total_profit = sum(sample_profit)  # Calculate the example profit KPI.
profit_margin = total_profit / total_sales if total_sales else None  # Calculate the ratio safely.
print("Total Sales:", total_sales)  # Display the sales KPI.
print("Total Profit:", total_profit)  # Display the profit KPI.
print("Profit Margin:", f"{profit_margin:.1%}" if profit_margin is not None else "N/A")  # Display the margin KPI.